## Setup: install required packages for this notebook

If imports fail (e.g., ModuleNotFoundError for scikit-learn), run the next cell to install dependencies into the active kernel. After installation, you may need to restart the kernel.

In [1]:
# Install dependencies (run if you see ModuleNotFoundError)
import sys, subprocess
pkgs = [
    'scikit-learn',
    'xgboost',
    'ta',
    'plotly',
    'nbformat'
]
for p in pkgs:
    try:
        __import__(p.replace('-', '_'))
    except Exception:
        print(f'Installing {p}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])
print('Done. If imports still fail, restart the kernel and rerun.')

Installing scikit-learn...
Done. If imports still fail, restart the kernel and rerun.
Done. If imports still fail, restart the kernel and rerun.


# Model Comparison Presentation
This notebook compares multiple models (XGBoost, ANN/MLP, SVM, Naive Bayes) on both regression and classification tasks using your existing src pipeline.
- Uses TimeSeriesSplit (5 folds) and reports per-fold metrics + summary.
- Loads a single ticker and horizon to keep the demo fast.
- Reuses feature engineering from `src.features` and labeling from `src.labeling`.

Tip: Run the cells in order. Adjust the ticker, date range, and horizon in the next cell if needed.

In [2]:
# Imports & settings
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score)
from xgboost import XGBRegressor, XGBClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.svm import SVR, SVC
from sklearn.naive_bayes import GaussianNB

# make sure we can import src/* when running this notebook
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    # if running from reports/ ensure project root is in path
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features import add_basic_features
from src.labeling import create_reg_target, create_class_labels

np.random.seed(42)

# Config (edit here for your demo)
TICKERS = ['BAC', 'NKE', 'TSLA']
START = '2018-01-01'
END = None  # to present
HORIZON = 1  # 1 day ahead
N_SPLITS = 5

# Ensure date boundaries are UTC-aware Timestamps to avoid tz-naive/aware comparison errors
START_DT = pd.to_datetime(START, errors='coerce', utc=True)
END_DT = pd.Timestamp.now(tz='UTC') if END is None else pd.to_datetime(END, errors='coerce', utc=True)


def prepare_numeric(df):
    # coerce key columns to numeric if present
    for c in ['Open','High','Low','Close','Volume']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

In [3]:
# Configure Plotly renderer for VS Code notebooks
import plotly.io as pio
pio.renderers.default = 'vscode'

In [4]:
# Load, engineer, and evaluate REGRESSION for each ticker
reg_results_all = {}
for TICKER in TICKERS:
    DATA_CSV = ROOT / 'data' / f'{TICKER}.csv'
    assert DATA_CSV.exists(), f'Missing data file: {DATA_CSV}'

    # Load & prepare
    df = pd.read_csv(DATA_CSV)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
    df = df[(df['Date'] >= START_DT) & (df['Date'] <= END_DT)].reset_index(drop=True)
    df = prepare_numeric(df)

    # Features & target
    df_reg = create_reg_target(df.copy(), horizon=HORIZON)
    df_reg = add_basic_features(df_reg)
    non_features_reg = {'Date','next_close'}
    features_reg = [c for c in df_reg.columns if c not in non_features_reg and np.issubdtype(df_reg[c].dtype, np.number)]
    df_reg = df_reg.dropna(subset=features_reg + ['next_close']).reset_index(drop=True)
    Xr = df_reg[features_reg]
    yr = df_reg['next_close']

    display(pd.DataFrame({'ticker':[TICKER], 'rows':[len(df_reg)], 'n_features':[len(features_reg)]}))

    # Evaluate regressors
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    reg_models = {
        'XGBRegressor': Pipeline([('scaler', StandardScaler()), ('xgb', XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, random_state=42))]),
        'MLPRegressor': Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))]),
        'SVR': Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
    }

    reg_results = {}
    for name, pipe in reg_models.items():
        fold_rows = []
        for i, (tr, te) in enumerate(tscv.split(Xr), start=1):
            Xtr, Xte = Xr.iloc[tr], Xr.iloc[te]
            ytr, yte = yr.iloc[tr], yr.iloc[te]
            # Fit and predict
            pipe.fit(Xtr, ytr)
            pred = pipe.predict(Xte)
            # Naive baseline: predict current Close for next_close
            baseline_pred = df_reg['Close'].iloc[te].to_numpy()
            # Fold test window
            test_start = pd.to_datetime(df_reg['Date'].iloc[te].min())
            test_end = pd.to_datetime(df_reg['Date'].iloc[te].max())
            # Metrics
            mae = float(mean_absolute_error(yte, pred))
            rmse = float(np.sqrt(mean_squared_error(yte, pred)))
            r2 = float(r2_score(yte, pred))
            mae_base = float(mean_absolute_error(yte, baseline_pred))
            rmse_base = float(np.sqrt(mean_squared_error(yte, baseline_pred)))
            fold_rows.append({
                'ticker': TICKER,
                'model': name,
                'fold': i,
                'test_start': str(test_start.date()) if not pd.isna(test_start) else None,
                'test_end': str(test_end.date()) if not pd.isna(test_end) else None,
                'mae': mae,
                'rmse': rmse,
                'r2': r2,
                'mae_base': mae_base,
                'rmse_base': rmse_base
            })
        res_df = pd.DataFrame(fold_rows)
        reg_results[name] = res_df
        summary_mean = res_df.agg({'mae':'mean','rmse':'mean','r2':'mean','mae_base':'mean','rmse_base':'mean'}).to_frame().T.assign(ticker=TICKER, model=name, fold='mean')
        summary_median = res_df.agg({'mae':'median','rmse':'median','r2':'median','mae_base':'median','rmse_base':'median'}).to_frame().T.assign(ticker=TICKER, model=name, fold='median')
        display(pd.concat([res_df, summary_mean, summary_median], ignore_index=True))
    reg_results_all[TICKER] = reg_results

,ticker,rows,n_features
0,BAC,1887,36


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,BAC,XGBRegressor,1,2019-05-24,2020-08-20,1.365163,1.854557,0.812489,0.519490,0.766708
1,BAC,XGBRegressor,2,2020-08-21,2021-11-17,4.032436,5.487446,0.420947,0.475032,0.614704
2,BAC,XGBRegressor,3,2021-11-18,2023-02-17,0.781059,1.031225,0.961254,0.551274,0.731357
3,BAC,XGBRegressor,4,2023-02-21,2024-05-20,0.443705,0.578133,0.973644,0.370032,0.496849
4,BAC,XGBRegressor,5,2024-05-21,2025-08-21,0.590101,0.818945,0.939811,0.486720,0.720234
5,BAC,XGBRegressor,mean,NaN,NaN,1.442493,1.954061,0.821629,0.480510,0.665970
6,BAC,XGBRegressor,median,NaN,NaN,0.781059,1.031225,0.939811,0.486720,0.720234


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning:

Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.



,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,BAC,MLPRegressor,1,2019-05-24,2020-08-20,12.816700,17.673329,-16.028802,0.519490,0.766708
1,BAC,MLPRegressor,2,2020-08-21,2021-11-17,4.235475,5.728367,0.368986,0.475032,0.614704
2,BAC,MLPRegressor,3,2021-11-18,2023-02-17,2.215585,2.818170,0.710626,0.551274,0.731357
3,BAC,MLPRegressor,4,2023-02-21,2024-05-20,1.444102,1.852423,0.729417,0.370032,0.496849
4,BAC,MLPRegressor,5,2024-05-21,2025-08-21,1.192679,1.672032,0.749103,0.486720,0.720234
5,BAC,MLPRegressor,mean,NaN,NaN,4.380908,5.948864,-2.694134,0.480510,0.665970
6,BAC,MLPRegressor,median,NaN,NaN,2.215585,2.818170,0.710626,0.486720,0.720234


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,BAC,SVR,1,2019-05-24,2020-08-20,2.708818,3.536388,0.318184,0.519490,0.766708
1,BAC,SVR,2,2020-08-21,2021-11-17,7.441267,10.068331,-0.949362,0.475032,0.614704
2,BAC,SVR,3,2021-11-18,2023-02-17,2.422407,3.640999,0.516979,0.551274,0.731357
3,BAC,SVR,4,2023-02-21,2024-05-20,0.514133,0.652099,0.966469,0.370032,0.496849
4,BAC,SVR,5,2024-05-21,2025-08-21,1.691793,2.571993,0.406329,0.486720,0.720234
5,BAC,SVR,mean,NaN,NaN,2.955684,4.093962,0.251720,0.480510,0.665970
6,BAC,SVR,median,NaN,NaN,2.422407,3.536388,0.406329,0.486720,0.720234


...

,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,NKE,SVR,1,2019-05-24,2020-08-20,11.433128,14.272992,-2.017508,1.377293,1.984473
1,NKE,SVR,2,2020-08-21,2021-11-17,57.239266,60.100750,-12.926964,1.620764,2.405562
2,NKE,SVR,3,2021-11-18,2023-02-17,8.515307,12.096038,0.688770,2.150541,2.854054
3,NKE,SVR,4,2023-02-21,2024-05-20,2.786522,3.786225,0.859721,1.267325,1.823898
4,NKE,SVR,5,2024-05-21,2025-08-21,19.707612,25.117465,-5.767423,1.137771,1.926325
5,NKE,SVR,mean,NaN,NaN,19.936367,23.074694,-3.832681,1.510739,2.198862
6,NKE,SVR,median,NaN,NaN,11.433128,14.272992,-2.017508,1.377293,1.984473


,ticker,rows,n_features
0,TSLA,1887,36


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,TSLA,XGBRegressor,1,2019-05-24,2020-08-20,19.546900,32.469354,-0.321210,1.424558,2.542127
1,TSLA,XGBRegressor,2,2020-08-21,2021-11-17,97.855075,112.967087,-2.855561,5.976918,8.565387
2,TSLA,XGBRegressor,3,2021-11-18,2023-02-17,12.280327,15.829704,0.946057,8.279693,10.994704
3,TSLA,XGBRegressor,4,2023-02-21,2024-05-20,5.759069,7.667109,0.956436,5.007994,6.783017
4,TSLA,XGBRegressor,5,2024-05-21,2025-08-21,15.051636,23.341207,0.890433,9.059204,12.444431
5,TSLA,XGBRegressor,mean,NaN,NaN,30.098602,38.454892,-0.076769,5.949673,8.265933
6,TSLA,XGBRegressor,median,NaN,NaN,15.051636,23.341207,0.890433,5.976918,8.565387


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,TSLA,MLPRegressor,1,2019-05-24,2020-08-20,76.172740,122.008082,-17.655277,1.424558,2.542127
1,TSLA,MLPRegressor,2,2020-08-21,2021-11-17,39.732818,43.958552,0.416191,5.976918,8.565387
2,TSLA,MLPRegressor,3,2021-11-18,2023-02-17,19.433309,24.017201,0.875826,8.279693,10.994704
3,TSLA,MLPRegressor,4,2023-02-21,2024-05-20,14.161391,17.189697,0.781020,5.007994,6.783017
4,TSLA,MLPRegressor,5,2024-05-21,2025-08-21,12.688055,16.983454,0.941992,9.059204,12.444431
5,TSLA,MLPRegressor,mean,NaN,NaN,32.437663,44.831397,-2.928050,5.949673,8.265933
6,TSLA,MLPRegressor,median,NaN,NaN,19.433309,24.017201,0.781020,5.976918,8.565387


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,TSLA,SVR,1,2019-05-24,2020-08-20,21.777792,34.204646,-0.466205,1.424558,2.542127
1,TSLA,SVR,2,2020-08-21,2021-11-17,184.859085,193.728796,-10.338929,5.976918,8.565387
2,TSLA,SVR,3,2021-11-18,2023-02-17,101.902993,119.599141,-2.079236,8.279693,10.994704
3,TSLA,SVR,4,2023-02-21,2024-05-20,11.022205,15.789664,0.815237,5.007994,6.783017
4,TSLA,SVR,5,2024-05-21,2025-08-21,70.862794,96.800978,-0.884487,9.059204,12.444431
5,TSLA,SVR,mean,NaN,NaN,78.084974,92.024645,-2.590724,5.949673,8.265933
6,TSLA,SVR,median,NaN,NaN,70.862794,96.800978,-0.884487,5.976918,8.565387


In [5]:
# Evaluate REGRESSORS (XGB, ANN/MLP, SVM) with 5-fold TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
reg_models = {
    'XGBRegressor': Pipeline([('scaler', StandardScaler()), ('xgb', XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, random_state=42))]),
    'MLPRegressor': Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))]),
    'SVR': Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
}

reg_results = {}
for name, pipe in reg_models.items():
    fold_rows = []
    for i, (tr, te) in enumerate(tscv.split(Xr), start=1):
        Xtr, Xte = Xr.iloc[tr], Xr.iloc[te]
        ytr, yte = yr.iloc[tr], yr.iloc[te]
        # Fit and predict
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        # Naive baseline: predict current Close for next_close
        baseline_pred = df_reg['Close'].iloc[te].to_numpy()
        # Fold test window
        test_start = pd.to_datetime(df_reg['Date'].iloc[te].min())
        test_end = pd.to_datetime(df_reg['Date'].iloc[te].max())
        # Metrics
        mae = float(mean_absolute_error(yte, pred))
        rmse = float(np.sqrt(mean_squared_error(yte, pred)))
        r2 = float(r2_score(yte, pred))
        mae_base = float(mean_absolute_error(yte, baseline_pred))
        rmse_base = float(np.sqrt(mean_squared_error(yte, baseline_pred)))
        fold_rows.append({
            'fold': i,
            'test_start': str(test_start.date()) if not pd.isna(test_start) else None,
            'test_end': str(test_end.date()) if not pd.isna(test_end) else None,
            'mae': mae,
            'rmse': rmse,
            'r2': r2,
            'mae_base': mae_base,
            'rmse_base': rmse_base
        })
    res_df = pd.DataFrame(fold_rows)
    reg_results[name] = res_df
    summary_mean = res_df.agg({'mae':'mean','rmse':'mean','r2':'mean','mae_base':'mean','rmse_base':'mean'}).to_frame().T.assign(fold='mean')
    summary_median = res_df.agg({'mae':'median','rmse':'median','r2':'median','mae_base':'median','rmse_base':'median'}).to_frame().T.assign(fold='median')
    display(pd.concat([res_df, summary_mean, summary_median], ignore_index=True))

,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,1,2019-05-24,2020-08-20,19.546900,32.469354,-0.321210,1.424558,2.542127
1,2,2020-08-21,2021-11-17,97.855075,112.967087,-2.855561,5.976918,8.565387
2,3,2021-11-18,2023-02-17,12.280327,15.829704,0.946057,8.279693,10.994704
3,4,2023-02-21,2024-05-20,5.759069,7.667109,0.956436,5.007994,6.783017
4,5,2024-05-21,2025-08-21,15.051636,23.341207,0.890433,9.059204,12.444431
5,mean,NaN,NaN,30.098602,38.454892,-0.076769,5.949673,8.265933
6,median,NaN,NaN,15.051636,23.341207,0.890433,5.976918,8.565387


,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,1,2019-05-24,2020-08-20,76.172740,122.008082,-17.655277,1.424558,2.542127
1,2,2020-08-21,2021-11-17,39.732818,43.958552,0.416191,5.976918,8.565387
2,3,2021-11-18,2023-02-17,19.433309,24.017201,0.875826,8.279693,10.994704
3,4,2023-02-21,2024-05-20,14.161391,17.189697,0.781020,5.007994,6.783017
4,5,2024-05-21,2025-08-21,12.688055,16.983454,0.941992,9.059204,12.444431
5,mean,NaN,NaN,32.437663,44.831397,-2.928050,5.949673,8.265933
6,median,NaN,NaN,19.433309,24.017201,0.781020,5.976918,8.565387


,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,1,2019-05-24,2020-08-20,21.777792,34.204646,-0.466205,1.424558,2.542127
1,2,2020-08-21,2021-11-17,184.859085,193.728796,-10.338929,5.976918,8.565387
2,3,2021-11-18,2023-02-17,101.902993,119.599141,-2.079236,8.279693,10.994704
3,4,2023-02-21,2024-05-20,11.022205,15.789664,0.815237,5.007994,6.783017
4,5,2024-05-21,2025-08-21,70.862794,96.800978,-0.884487,9.059204,12.444431
5,mean,NaN,NaN,78.084974,92.024645,-2.590724,5.949673,8.265933
6,median,NaN,NaN,70.862794,96.800978,-0.884487,5.976918,8.565387


In [6]:
# Load, engineer, and evaluate CLASSIFICATION for each ticker
cls_results_all = {}
for TICKER in TICKERS:
    DATA_CSV = ROOT / 'data' / f'{TICKER}.csv'
    assert DATA_CSV.exists(), f'Missing data file: {DATA_CSV}'

    # Use already-loaded df from regression loop if ticker same? For clarity, reload.
    df = pd.read_csv(DATA_CSV)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
    df = df[(df['Date'] >= START_DT) & (df['Date'] <= END_DT)].reset_index(drop=True)
    df = prepare_numeric(df)

    dfc = create_class_labels(df.copy(), horizon=HORIZON)
    dfc = add_basic_features(dfc)
    non_features_cls = {'Date','next_close','ret_next','label'}
    features_cls = [c for c in dfc.columns if c not in non_features_cls and np.issubdtype(dfc[c].dtype, np.number)]
    dfc = dfc.dropna(subset=features_cls + ['label']).reset_index(drop=True)
    Xc = dfc[features_cls]
    yc = (dfc['label'] + 1).astype(int)  # -1,0,1 -> 0,1,2

    display(pd.DataFrame({'ticker':[TICKER], 'rows':[len(dfc)], 'n_features':[len(features_cls)]}))

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    cls_models = {
        'XGBClassifier': Pipeline([('scaler', StandardScaler()), ('xgb', XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, use_label_encoder=False, eval_metric='mlogloss', random_state=42))]),
        'MLPClassifier': Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))]),
        'SVC': Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True, random_state=42))]),
        'GaussianNB': Pipeline([('scaler', StandardScaler()), ('nb', GaussianNB())])
    }

    cls_results = {}
    for name, pipe in cls_models.items():
        fold_rows = []
        for i, (tr, te) in enumerate(tscv.split(Xc), start=1):
            Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
            ytr, yte = yc.iloc[tr], yc.iloc[te]
            pipe.fit(Xtr, ytr)
            pred = pipe.predict(Xte)
            # back to -1/0/1
            pred_orig = pred - 1
            yte_orig = yte - 1
            # Fold test window
            test_start = pd.to_datetime(dfc['Date'].iloc[te].min())
            test_end = pd.to_datetime(dfc['Date'].iloc[te].max())
            fold_rows.append({
                'ticker': TICKER,
                'model': name,
                'fold': i,
                'test_start': str(test_start.date()) if not pd.isna(test_start) else None,
                'test_end': str(test_end.date()) if not pd.isna(test_end) else None,
                'accuracy': float(accuracy_score(yte_orig, pred_orig)),
                'precision_macro': float(precision_score(yte_orig, pred_orig, average='macro', zero_division=0)),
                'recall_macro': float(recall_score(yte_orig, pred_orig, average='macro', zero_division=0)),
                'f1_macro': float(f1_score(yte_orig, pred_orig, average='macro', zero_division=0))
            })
        res_df = pd.DataFrame(fold_rows)
        cls_results[name] = res_df
        summary_mean = res_df.agg({'accuracy':'mean','precision_macro':'mean','recall_macro':'mean','f1_macro':'mean'}).to_frame().T.assign(ticker=TICKER, model=name, fold='mean')
        summary_median = res_df.agg({'accuracy':'median','precision_macro':'median','recall_macro':'median','f1_macro':'median'}).to_frame().T.assign(ticker=TICKER, model=name, fold='median')
        display(pd.concat([res_df, summary_mean, summary_median], ignore_index=True))
    cls_results_all[TICKER] = cls_results

,ticker,rows,n_features
0,BAC,1887,36


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:23:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:23:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:23:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3

,ticker,model,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,BAC,XGBClassifier,1,2019-05-24,2020-08-20,0.356688,0.363909,0.355931,0.353553
1,BAC,XGBClassifier,2,2020-08-21,2021-11-17,0.372611,0.386129,0.395696,0.372627
2,BAC,XGBClassifier,3,2021-11-18,2023-02-17,0.328025,0.318388,0.321085,0.314455
3,BAC,XGBClassifier,4,2023-02-21,2024-05-20,0.334395,0.327475,0.328109,0.326052
4,BAC,XGBClassifier,5,2024-05-21,2025-08-21,0.343949,0.352685,0.343775,0.337429
5,BAC,XGBClassifier,mean,NaN,NaN,0.347134,0.349717,0.348919,0.340823
6,BAC,XGBClassifier,median,NaN,NaN,0.343949,0.352685,0.343775,0.337429


,ticker,model,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,BAC,MLPClassifier,1,2019-05-24,2020-08-20,0.385350,0.341633,0.352634,0.337135
1,BAC,MLPClassifier,2,2020-08-21,2021-11-17,0.305732,0.327361,0.330370,0.304177
2,BAC,MLPClassifier,3,2021-11-18,2023-02-17,0.353503,0.337082,0.335284,0.334552
3,BAC,MLPClassifier,4,2023-02-21,2024-05-20,0.343949,0.342085,0.335633,0.314986
4,BAC,MLPClassifier,5,2024-05-21,2025-08-21,0.337580,0.367334,0.330303,0.297708
5,BAC,MLPClassifier,mean,NaN,NaN,0.345223,0.343099,0.336845,0.317712
6,BAC,MLPClassifier,median,NaN,NaN,0.343949,0.341633,0.335284,0.314986


,ticker,model,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,BAC,SVC,1,2019-05-24,2020-08-20,0.423567,0.414360,0.417571,0.408416
1,BAC,SVC,2,2020-08-21,2021-11-17,0.343949,0.322735,0.324934,0.315351
2,BAC,SVC,3,2021-11-18,2023-02-17,0.321656,0.350947,0.317727,0.255018
3,BAC,SVC,4,2023-02-21,2024-05-20,0.378981,0.374172,0.365914,0.347333
4,BAC,SVC,5,2024-05-21,2025-08-21,0.350318,0.349916,0.351712,0.320913
5,BAC,SVC,mean,NaN,NaN,0.363694,0.362426,0.355572,0.329406
6,BAC,SVC,median,NaN,NaN,0.350318,0.350947,0.351712,0.320913


...

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:23:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:23:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:23:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3

,ticker,model,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,TSLA,XGBClassifier,1,2019-05-24,2020-08-20,0.382166,0.381399,0.356546,0.301617
1,TSLA,XGBClassifier,2,2020-08-21,2021-11-17,0.404459,0.368821,0.314310,0.290823
2,TSLA,XGBClassifier,3,2021-11-18,2023-02-17,0.449045,0.395626,0.383443,0.385831
3,TSLA,XGBClassifier,4,2023-02-21,2024-05-20,0.401274,0.400865,0.323858,0.312510
4,TSLA,XGBClassifier,5,2024-05-21,2025-08-21,0.423567,0.285686,0.329740,0.305552
5,TSLA,XGBClassifier,mean,NaN,NaN,0.412102,0.366480,0.341579,0.319267
6,TSLA,XGBClassifier,median,NaN,NaN,0.404459,0.381399,0.329740,0.305552


,ticker,model,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,TSLA,MLPClassifier,1,2019-05-24,2020-08-20,0.407643,0.285034,0.342718,0.292792
1,TSLA,MLPClassifier,2,2020-08-21,2021-11-17,0.458599,0.304540,0.339933,0.267288
2,TSLA,MLPClassifier,3,2021-11-18,2023-02-17,0.461783,0.308999,0.355686,0.325047
3,TSLA,MLPClassifier,4,2023-02-21,2024-05-20,0.420382,0.283338,0.329742,0.303917
4,TSLA,MLPClassifier,5,2024-05-21,2025-08-21,0.464968,0.309995,0.361950,0.333341
5,TSLA,MLPClassifier,mean,NaN,NaN,0.442675,0.298381,0.346006,0.304477
6,TSLA,MLPClassifier,median,NaN,NaN,0.458599,0.304540,0.342718,0.303917


,ticker,model,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,TSLA,SVC,1,2019-05-24,2020-08-20,0.350318,0.270383,0.335547,0.195950
1,TSLA,SVC,2,2020-08-21,2021-11-17,0.458599,0.152866,0.333333,0.209607
2,TSLA,SVC,3,2021-11-18,2023-02-17,0.452229,0.301525,0.346970,0.322644
3,TSLA,SVC,4,2023-02-21,2024-05-20,0.464968,0.312597,0.364942,0.328108
4,TSLA,SVC,5,2024-05-21,2025-08-21,0.464968,0.309884,0.361931,0.333514
5,TSLA,SVC,mean,NaN,NaN,0.438217,0.269451,0.348545,0.277965
6,TSLA,SVC,median,NaN,NaN,0.458599,0.301525,0.346970,0.322644


,ticker,model,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,TSLA,GaussianNB,1,2019-05-24,2020-08-20,0.363057,0.383494,0.363079,0.262827
1,TSLA,GaussianNB,2,2020-08-21,2021-11-17,0.458599,0.152866,0.333333,0.209607
2,TSLA,GaussianNB,3,2021-11-18,2023-02-17,0.423567,0.508228,0.324766,0.215794
3,TSLA,GaussianNB,4,2023-02-21,2024-05-20,0.378981,0.203843,0.348691,0.254931
4,TSLA,GaussianNB,5,2024-05-21,2025-08-21,0.401274,0.210219,0.350617,0.255084
5,TSLA,GaussianNB,mean,NaN,NaN,0.405096,0.291730,0.344097,0.239649
6,TSLA,GaussianNB,median,NaN,NaN,0.401274,0.210219,0.348691,0.254931


In [7]:
# Evaluate CLASSIFIERS (XGB, ANN/MLP, SVM, Naive Bayes) with 5-fold TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
cls_models = {
    'XGBClassifier': Pipeline([('scaler', StandardScaler()), ('xgb', XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, use_label_encoder=False, eval_metric='mlogloss', random_state=42))]),
    'MLPClassifier': Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))]),
    'SVC': Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True, random_state=42))]),
    'GaussianNB': Pipeline([('scaler', StandardScaler()), ('nb', GaussianNB())])
}

cls_results = {}
for name, pipe in cls_models.items():
    fold_rows = []
    for i, (tr, te) in enumerate(tscv.split(Xc), start=1):
        Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
        ytr, yte = yc.iloc[tr], yc.iloc[te]
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        # back to -1/0/1
        pred_orig = pred - 1
        yte_orig = yte - 1
        # Fold test window
        test_start = pd.to_datetime(dfc['Date'].iloc[te].min())
        test_end = pd.to_datetime(dfc['Date'].iloc[te].max())
        fold_rows.append({
            'fold': i,
            'test_start': str(test_start.date()) if not pd.isna(test_start) else None,
            'test_end': str(test_end.date()) if not pd.isna(test_end) else None,
            'accuracy': float(accuracy_score(yte_orig, pred_orig)),
            'precision_macro': float(precision_score(yte_orig, pred_orig, average='macro', zero_division=0)),
            'recall_macro': float(recall_score(yte_orig, pred_orig, average='macro', zero_division=0)),
            'f1_macro': float(f1_score(yte_orig, pred_orig, average='macro', zero_division=0))
        })
    res_df = pd.DataFrame(fold_rows)
    cls_results[name] = res_df
    summary_mean = res_df.agg({'accuracy':'mean','precision_macro':'mean','recall_macro':'mean','f1_macro':'mean'}).to_frame().T.assign(fold='mean')
    summary_median = res_df.agg({'accuracy':'median','precision_macro':'median','recall_macro':'median','f1_macro':'median'}).to_frame().T.assign(fold='median')
    display(pd.concat([res_df, summary_mean, summary_median], ignore_index=True))

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:24:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:24:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:24:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3

,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,1,2019-05-24,2020-08-20,0.382166,0.381399,0.356546,0.301617
1,2,2020-08-21,2021-11-17,0.404459,0.368821,0.314310,0.290823
2,3,2021-11-18,2023-02-17,0.449045,0.395626,0.383443,0.385831
3,4,2023-02-21,2024-05-20,0.401274,0.400865,0.323858,0.312510
4,5,2024-05-21,2025-08-21,0.423567,0.285686,0.329740,0.305552
5,mean,NaN,NaN,0.412102,0.366480,0.341579,0.319267
6,median,NaN,NaN,0.404459,0.381399,0.329740,0.305552


,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,1,2019-05-24,2020-08-20,0.407643,0.285034,0.342718,0.292792
1,2,2020-08-21,2021-11-17,0.458599,0.304540,0.339933,0.267288
2,3,2021-11-18,2023-02-17,0.461783,0.308999,0.355686,0.325047
3,4,2023-02-21,2024-05-20,0.420382,0.283338,0.329742,0.303917
4,5,2024-05-21,2025-08-21,0.464968,0.309995,0.361950,0.333341
5,mean,NaN,NaN,0.442675,0.298381,0.346006,0.304477
6,median,NaN,NaN,0.458599,0.304540,0.342718,0.303917


,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,1,2019-05-24,2020-08-20,0.350318,0.270383,0.335547,0.195950
1,2,2020-08-21,2021-11-17,0.458599,0.152866,0.333333,0.209607
2,3,2021-11-18,2023-02-17,0.452229,0.301525,0.346970,0.322644
3,4,2023-02-21,2024-05-20,0.464968,0.312597,0.364942,0.328108
4,5,2024-05-21,2025-08-21,0.464968,0.309884,0.361931,0.333514
5,mean,NaN,NaN,0.438217,0.269451,0.348545,0.277965
6,median,NaN,NaN,0.458599,0.301525,0.346970,0.322644


,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,1,2019-05-24,2020-08-20,0.363057,0.383494,0.363079,0.262827
1,2,2020-08-21,2021-11-17,0.458599,0.152866,0.333333,0.209607
2,3,2021-11-18,2023-02-17,0.423567,0.508228,0.324766,0.215794
3,4,2023-02-21,2024-05-20,0.378981,0.203843,0.348691,0.254931
4,5,2024-05-21,2025-08-21,0.401274,0.210219,0.350617,0.255084
5,mean,NaN,NaN,0.405096,0.291730,0.344097,0.239649
6,median,NaN,NaN,0.401274,0.210219,0.348691,0.254931


## Notes
- These runs avoid GridSearchCV to keep the demo fast. For your final report, you can cite GridSearchCV results from the app logs.
- Use the same date range and horizon as your Streamlit app to align results.
- You can extend this notebook to plot predictions vs actuals or confusion matrices as needed.

## Aggregate results and export

We’ll combine per-fold results across tickers/models into unified tables and save CSVs for regression and classification.

In [8]:
# Aggregate and export results
import pandas as pd
from pathlib import Path

# Collect regression results
reg_tables = []
if 'reg_results_all' in globals():
    for ticker, models in reg_results_all.items():
        for model, df_res in models.items():
            # df_res already contains ticker/model columns in our latest version; enforce just in case
            if 'ticker' not in df_res.columns:
                df_res = df_res.assign(ticker=ticker, model=model)
            reg_tables.append(df_res)
reg_all_df = pd.concat(reg_tables, ignore_index=True) if reg_tables else pd.DataFrame()

# Collect classification results
cls_tables = []
if 'cls_results_all' in globals():
    for ticker, models in cls_results_all.items():
        for model, df_res in models.items():
            if 'ticker' not in df_res.columns:
                df_res = df_res.assign(ticker=ticker, model=model)
            cls_tables.append(df_res)
cls_all_df = pd.concat(cls_tables, ignore_index=True) if cls_tables else pd.DataFrame()

# Display small previews
if not reg_all_df.empty:
    display(reg_all_df.head())
if not cls_all_df.empty:
    display(cls_all_df.head())

# Optional: export to CSV under reports/
EXPORT = True
out_dir = Path.cwd().parent / 'reports'
out_dir.mkdir(exist_ok=True)
if EXPORT:
    if not reg_all_df.empty:
        reg_out = out_dir / 'regression_results.csv'
        reg_all_df.to_csv(reg_out, index=False)
        print(f'Regression results saved to: {reg_out}')
    if not cls_all_df.empty:
        cls_out = out_dir / 'classification_results.csv'
        cls_all_df.to_csv(cls_out, index=False)
        print(f'Classification results saved to: {cls_out}')

,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,BAC,XGBRegressor,1,2019-05-24,2020-08-20,1.365163,1.854557,0.812489,0.519490,0.766708
1,BAC,XGBRegressor,2,2020-08-21,2021-11-17,4.032436,5.487446,0.420947,0.475032,0.614704
2,BAC,XGBRegressor,3,2021-11-18,2023-02-17,0.781059,1.031225,0.961254,0.551274,0.731357
3,BAC,XGBRegressor,4,2023-02-21,2024-05-20,0.443705,0.578133,0.973644,0.370032,0.496849
4,BAC,XGBRegressor,5,2024-05-21,2025-08-21,0.590101,0.818945,0.939811,0.486720,0.720234


,ticker,model,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,BAC,XGBClassifier,1,2019-05-24,2020-08-20,0.356688,0.363909,0.355931,0.353553
1,BAC,XGBClassifier,2,2020-08-21,2021-11-17,0.372611,0.386129,0.395696,0.372627
2,BAC,XGBClassifier,3,2021-11-18,2023-02-17,0.328025,0.318388,0.321085,0.314455
3,BAC,XGBClassifier,4,2023-02-21,2024-05-20,0.334395,0.327475,0.328109,0.326052
4,BAC,XGBClassifier,5,2024-05-21,2025-08-21,0.343949,0.352685,0.343775,0.337429


Regression results saved to: c:\Users\user\Desktop\Project\StockMaster\reports\regression_results.csv
Classification results saved to: c:\Users\user\Desktop\Project\StockMaster\reports\classification_results.csv


## Quick plots

We’ll show simple predicted vs actual plots for regression and confusion matrices for classification per ticker/model.

In [9]:
# Plot predicted vs actual for regression (last fold per ticker/model)
import plotly.graph_objects as go
import plotly.io as pio
# Ensure nbformat is available for MIME rendering in notebooks
try:
    import nbformat  # noqa: F401
except Exception:
    import sys, subprocess
    print('Installing nbformat for Plotly rendering...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nbformat'])
# Set renderer for VS Code notebooks
pio.renderers.default = 'vscode'

if 'reg_results_all' in globals():
    for ticker, models in reg_results_all.items():
        for model, df_res in models.items():
            # Need predictions to plot; recompute quickly for the last fold
            DATA_CSV = Path.cwd().parent / 'data' / f'{ticker}.csv'
            df = pd.read_csv(DATA_CSV)
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
            df = df[(df['Date'] >= START_DT) & (df['Date'] <= END_DT)].reset_index(drop=True)
            df = prepare_numeric(df)
            df_reg = create_reg_target(df.copy(), horizon=HORIZON)
            df_reg = add_basic_features(df_reg)
            non_features_reg = {'Date','next_close'}
            features_reg = [c for c in df_reg.columns if c not in non_features_reg and np.issubdtype(df_reg[c].dtype, np.number)]
            df_reg = df_reg.dropna(subset=features_reg + ['next_close']).reset_index(drop=True)
            Xr = df_reg[features_reg]
            yr = df_reg['next_close']

            if model == 'XGBRegressor':
                pipe = Pipeline([('scaler', StandardScaler()), ('xgb', XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, random_state=42))])
            elif model == 'MLPRegressor':
                pipe = Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))])
            else:
                pipe = Pipeline([('scaler', StandardScaler()), ('svr', SVR())])

            tscv = TimeSeriesSplit(n_splits=N_SPLITS)
            last_split = list(tscv.split(Xr))[-1]
            tr, te = last_split
            pipe.fit(Xr.iloc[tr], yr.iloc[tr])
            pred = pipe.predict(Xr.iloc[te])
            dates = df_reg['Date'].iloc[te]
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=dates, y=yr.iloc[te], mode='lines', name='Actual'))
            fig.add_trace(go.Scatter(x=dates, y=pred, mode='lines', name='Predicted'))
            fig.update_layout(title=f'{ticker} - {model}: Predicted vs Actual (last fold)', xaxis_title='Date', yaxis_title='Price')
            fig.show()

In [10]:
# Plot confusion matrices for classification (last fold per ticker/model)
import numpy as np
import plotly.figure_factory as ff
import plotly.io as pio
# Ensure nbformat is available for MIME rendering in notebooks
try:
    import nbformat  # noqa: F401
except Exception:
    import sys, subprocess
    print('Installing nbformat for Plotly rendering...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nbformat'])
# Set renderer for VS Code notebooks
pio.renderers.default = 'vscode'

if 'cls_results_all' in globals():
    from sklearn.metrics import confusion_matrix
    for ticker, models in cls_results_all.items():
        for model, df_res in models.items():
            DATA_CSV = Path.cwd().parent / 'data' / f'{ticker}.csv'
            df = pd.read_csv(DATA_CSV)
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
            df = df[(df['Date'] >= START_DT) & (df['Date'] <= END_DT)].reset_index(drop=True)
            df = prepare_numeric(df)

            dfc = create_class_labels(df.copy(), horizon=HORIZON)
            dfc = add_basic_features(dfc)
            non_features_cls = {'Date','next_close','ret_next','label'}
            features_cls = [c for c in dfc.columns if c not in non_features_cls and np.issubdtype(dfc[c].dtype, np.number)]
            dfc = dfc.dropna(subset=features_cls + ['label']).reset_index(drop=True)
            Xc = dfc[features_cls]
            yc = (dfc['label'] + 1).astype(int)

            if model == 'XGBClassifier':
                pipe = Pipeline([('scaler', StandardScaler()), ('xgb', XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, use_label_encoder=False, eval_metric='mlogloss', random_state=42))])
            elif model == 'MLPClassifier':
                pipe = Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))])
            elif model == 'SVC':
                pipe = Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True, random_state=42))])
            else:
                pipe = Pipeline([('scaler', StandardScaler()), ('nb', GaussianNB())])

            tscv = TimeSeriesSplit(n_splits=N_SPLITS)
            tr, te = list(tscv.split(Xc))[-1]
            pipe.fit(Xc.iloc[tr], yc.iloc[tr])
            pred = pipe.predict(Xc.iloc[te])
            # Convert back to -1/0/1 labels
            pred_orig = pred - 1
            yte_orig = yc.iloc[te] - 1
            cm = confusion_matrix(yte_orig, pred_orig, labels=[-1,0,1])
            z = cm.astype(int)
            fig = ff.create_annotated_heatmap(z, x=['-1','0','1'], y=['-1','0','1'], colorscale='Blues')
            fig.update_layout(title=f'{ticker} - {model}: Confusion matrix (last fold)')
            fig.update_xaxes(title_text='Predicted')
            fig.update_yaxes(title_text='Actual')
            fig.show()

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:24:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.




C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:24:10] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.




C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning:

[23:24:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.


